In [10]:
from pathlib import Path
import pandas as pd

In [11]:
YEAR = 2023
PROJECT_DIR = Path("../..").resolve()

print("Base Directory:", PROJECT_DIR)


Base Directory: /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline


## Load Data

### Tennis is My Life

In [12]:


df_timl = pd.read_csv(PROJECT_DIR / f"data/raw/timl/atp/{YEAR}.csv")
print(list(df_timl.columns))
df_timl.head()



['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'indoor', 'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry', 'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age', 'winner_rank', 'winner_rank_points', 'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand', 'loser_ht', 'loser_ioc', 'loser_age', 'loser_rank', 'loser_rank_points', 'score', 'best_of', 'round', 'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced']


,tourney_id,tourney_name,surface,draw_size,tourney_level,indoor,tourney_date,match_num,winner_id,winner_seed,...,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced
0,2023-9900,United Cup,Hard,18,A,O,20230102,1,GH92,NaN,...,0.0,0.0,0.0,81.0,50.0,37.0,15.0,11.0,3.0,4.0
1,2023-9900,United Cup,Hard,18,A,O,20230102,2,CG80,8.0,...,10.0,2.0,1.0,70.0,50.0,31.0,8.0,11.0,5.0,9.0
2,2023-9900,United Cup,Hard,18,A,O,20230102,3,ME82,NaN,...,3.0,2.0,2.0,56.0,37.0,16.0,5.0,6.0,4.0,10.0
3,2023-9900,United Cup,Hard,18,A,O,20230102,4,RC91,6.0,...,0.0,1.0,3.0,57.0,37.0,22.0,10.0,9.0,4.0,7.0
4,2023-9900,United Cup,Hard,18,A,O,20230102,5,GH92,NaN,...,4.0,9.0,1.0,104.0,63.0,51.0,21.0,17.0,3.0,5.0


In [26]:
df_uk["year"] = YEAR

### Tennis Data UK

In [13]:
df_uk = pd.read_csv(f"../../data/clean/tennis-data-uk/atp/atp_singles_results_{YEAR}.csv")
print(list(df_uk.columns))

df_uk.head()

['ATP', 'Location', 'Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank', 'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W', 'B365L', 'PSW', 'PSL', 'MaxW', 'MaxL', 'AvgW', 'AvgL']


,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,Lsets,Comment,B365W,B365L,PSW,PSL,MaxW,MaxL,AvgW,AvgL
0,1,Adelaide,Adelaide International 1,1/1/23,ATP250,Outdoor,Hard,1st Round,3,Giron M.,...,1.0,Completed,1.91,1.91,1.93,1.95,1.99,1.95,1.89,1.89
1,1,Adelaide,Adelaide International 1,1/1/23,ATP250,Outdoor,Hard,1st Round,3,Mcdonald M.,...,0.0,Retired,1.36,3.20,1.39,3.25,1.44,3.40,1.36,3.12
2,1,Adelaide,Adelaide International 1,1/2/23,ATP250,Outdoor,Hard,1st Round,3,Kecmanovic M.,...,0.0,Completed,1.57,2.38,1.58,2.53,1.64,2.53,1.58,2.36
3,1,Adelaide,Adelaide International 1,1/2/23,ATP250,Outdoor,Hard,1st Round,3,Nishioka Y.,...,1.0,Completed,3.75,1.29,4.00,1.28,4.00,1.31,3.56,1.29
4,1,Adelaide,Adelaide International 1,1/2/23,ATP250,Outdoor,Hard,1st Round,3,Popyrin A.,...,0.0,Completed,6.50,1.11,6.20,1.15,6.75,1.18,6.04,1.13


Get Tournaments and their information

In [14]:
df_uk[["ATP", "Location", "Tournament", "Series", "Court", "Surface", "Best of"]].drop_duplicates(keep="first")

,ATP,Location,Tournament,Series,Court,Surface,Best of
0,1,Adelaide,Adelaide International 1,ATP250,Outdoor,Hard,3
31,2,Pune,Maharashtra Open,ATP250,Outdoor,Hard,3
58,3,Adelaide,Adelaide International 2,ATP250,Outdoor,Hard,3
85,4,Auckland,ASB Classic,ATP250,Outdoor,Hard,3
112,5,Melbourne,Australian Open,Grand Slam,Outdoor,Hard,5
...,...,...,...,...,...,...,...
2548,60,Vienna,Vienna Open,ATP500,Indoor,Hard,3
2579,61,Paris,BNP Paribas Masters,Masters 1000,Indoor,Hard,3
2634,62,Metz,Open de Moselle,ATP250,Indoor,Hard,3
2661,63,Sofia,Sofia Open,ATP250,Indoor,Hard,3


In [21]:
def find_uk_inconsistent_tournaments(
    df: pd.DataFrame,
    key_columns: list[str] | None = None,
    info_cols: list[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Find tournaments whose attributes aren't consistent across all their rows.

    Returns:
        metrics: distinct-value count per info column, one row per inconsistent key.
        affected_rows: the original df_uk rows belonging to those inconsistent keys,
            for manual review/fixing in the source data.
    """
    key_columns = key_columns or ["ATP", "Location"]
    info_cols = info_cols or ["Tournament", "Series", "Court", "Surface", "Best of"]

    nunique_per_key = df.groupby(key_columns)[info_cols].nunique()
    metrics = nunique_per_key[(nunique_per_key > 1).any(axis=1)]

    if metrics.empty:
        print("All tournament attributes are consistent.")
        return metrics, df.iloc[0:0]

    print(f"Warning: {len(metrics)} tournament(s) have inconsistent attributes:")

    affected_rows = df.merge(metrics.reset_index()[key_columns], on=key_columns, how="inner")

    return metrics, affected_rows


In [22]:
def find_uk_reused_tournament_ids(
    df: pd.DataFrame,
    id_col: str = "ATP",
    disambiguating_cols: list[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Find tournament ids reused across genuinely different tournaments.

    `find_uk_inconsistent_tournaments` keys on (ATP, Location), so same-week
    collisions like Stockholm/Tokyo both being "ATP 58" get silently split
    into two clean groups and never show up there. This checks `id_col` on
    its own instead.

    Returns:
        metrics: distinct-value count of `disambiguating_cols` per id, one
            row per id used by more than one tournament.
        affected_rows: the original df_uk rows for those reused ids.
    """
    disambiguating_cols = disambiguating_cols or ["Location", "Tournament"]

    nunique_per_id = df.groupby(id_col)[disambiguating_cols].nunique()
    metrics = nunique_per_id[(nunique_per_id > 1).any(axis=1)]

    if metrics.empty:
        print(f"Every {id_col} id maps to exactly one tournament.")
        return metrics, df.iloc[0:0]

    print(f"Warning: {len(metrics)} {id_col} id(s) are reused across different tournaments:")

    affected_rows = df.merge(metrics.reset_index()[[id_col]], on=id_col, how="inner")

    return metrics, affected_rows



In [20]:
reused_metrics, reused_rows = find_uk_reused_tournament_ids(df_uk)

display(reused_metrics)
display(reused_rows)


,Location,Tournament
ATP,,
58,2,2


,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,Lsets,Comment,B365W,B365L,PSW,PSL,MaxW,MaxL,AvgW,AvgL
0,58,Stockholm,Nordic Open,10/16/23,ATP250,Indoor,Hard,1st Round,3,Sonego L.,...,1.0,Completed,1.40,3.00,1.37,3.32,1.42,3.32,1.38,3.02
1,58,Stockholm,Nordic Open,10/16/23,ATP250,Indoor,Hard,1st Round,3,Prizmic D.,...,0.0,Completed,2.30,1.62,2.30,1.68,2.34,1.72,2.24,1.65
2,58,Stockholm,Nordic Open,10/16/23,ATP250,Indoor,Hard,1st Round,3,Kecmanovic M.,...,0.0,Completed,1.11,6.50,1.14,6.62,1.17,6.62,1.13,5.91
3,58,Stockholm,Nordic Open,10/16/23,ATP250,Indoor,Hard,1st Round,3,Ymer E.,...,0.0,Completed,2.63,1.50,2.62,1.55,2.67,1.58,2.53,1.51
4,58,Stockholm,Nordic Open,10/17/23,ATP250,Indoor,Hard,1st Round,3,Wolf J.J.,...,0.0,Completed,1.62,2.30,1.70,2.27,1.73,2.44,1.64,2.23
5,58,Stockholm,Nordic Open,10/17/23,ATP250,Indoor,Hard,1st Round,3,Kotov P.,...,1.0,Completed,1.91,1.91,1.81,2.10,2.00,2.10,1.86,1.93
6,58,Stockholm,Nordic Open,10/17/23,ATP250,Indoor,Hard,1st Round,3,Ruusuvuori E.,...,0.0,Completed,1.20,4.50,1.19,5.40,1.21,5.40,1.18,4.70
7,58,Stockholm,Nordic Open,10/17/23,ATP250,Indoor,Hard,1st Round,3,Misolic F.,...,0.0,Completed,4.00,1.25,4.43,1.24,4.43,1.31,3.88,1.25
8,58,Stockholm,Nordic Open,10/17/23,ATP250,Indoor,Hard,1st Round,3,Djere L.,...,0.0,Completed,1.14,5.50,1.16,5.95,1.19,6.19,1.15,5.42
9,58,Stockholm,Nordic Open,10/17/23,ATP250,Indoor,Hard,1st Round,3,Wawrinka S.,...,0.0,Completed,1.67,2.20,1.75,2.19,1.75,2.26,1.69,2.15


In [16]:
metrics, affected_rows = find_uk_inconsistent_tournaments(df_uk)

display(metrics)
display(affected_rows)


,,Tournament,Series,Court,Surface,Best of
ATP,Location,,,,,
38,London,1,1,1,1,2


,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,Lsets,Comment,B365W,B365L,PSW,PSL,MaxW,MaxL,AvgW,AvgL
0,38,London,Wimbledon,7/3/23,Grand Slam,Outdoor,Grass,1st Round,5,Barrios M.,...,1.0,Completed,2.10,1.73,1.99,1.92,2.20,1.92,2.08,1.77
1,38,London,Wimbledon,7/3/23,Grand Slam,Outdoor,Grass,1st Round,5,Musetti L.,...,0.0,Completed,1.06,10.00,1.07,11.93,1.08,13.50,1.05,10.47
2,38,London,Wimbledon,7/3/23,Grand Slam,Outdoor,Grass,1st Round,5,Karatsev A.,...,1.0,Completed,1.73,2.10,1.63,2.45,1.84,2.45,1.67,2.24
3,38,London,Wimbledon,7/3/23,Grand Slam,Outdoor,Grass,1st Round,5,Thompson J.,...,2.0,Completed,1.91,1.91,1.88,2.04,1.95,2.06,1.89,1.94
4,38,London,Wimbledon,7/3/23,Grand Slam,Outdoor,Grass,1st Round,5,Rublev A.,...,0.0,Completed,1.06,10.00,1.08,10.52,1.10,11.22,1.07,8.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,38,London,Wimbledon,7/12/23,Grand Slam,Outdoor,Grass,Quarterfinals,5,Medvedev D.,...,2.0,Completed,1.17,5.00,1.16,6.35,1.21,6.35,1.16,5.40
123,38,London,Wimbledon,7/12/23,Grand Slam,Outdoor,Grass,Quarterfinals,5,Alcaraz C.,...,0.0,Completed,1.22,4.33,1.23,4.70,1.29,4.82,1.23,4.29
124,38,London,Wimbledon,7/14/23,Grand Slam,Outdoor,Grass,Semifinals,5,Djokovic N.,...,0.0,Completed,1.20,4.50,1.25,4.44,1.28,4.60,1.23,4.36
125,38,London,Wimbledon,7/14/23,Grand Slam,Outdoor,Grass,Semifinals,5,Alcaraz C.,...,0.0,Completed,1.36,3.20,1.42,3.12,1.44,3.30,1.40,3.01


In [23]:
import re


def slugify(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def add_source_event_key(df):
    df = df.copy()

    df["year"] = df["Date"].dt.year

    df["source_event_key"] = (
        df["year"].astype(str)
        + "_"
        + df["ATP"].astype(str)
        + "_"
        + df["Location"].map(slugify)
        + "_"
        + df["Tournament"].map(slugify)
    )

    return df

In [27]:
add_source_event_key(df_uk)

AttributeError: Can only use .dt accessor with datetimelike values

In [24]:
add_source_event_key(df_uk)

AttributeError: Can only use .dt accessor with datetimelike values

In [ ]:
display(
    df_uk.loc[
        df_uk["ATP"].isin([38]),
        ["ATP", "Location", "Tournament", "Series", "Court", "Surface", "Best of", "Date"],
    ].drop_duplicates()
)


In [ ]:
uk_tourneys = df_uk['Tournament'].unique()
print(uk_tourneys)

In [ ]:
df_timl.loc[~df_timl['tourney_name'].str.contains("Davis Cup"), "tourney_name"].unique()

In [ ]:
source_df = df_uk.copy()
canonical_df = df_timl.loc[~df_timl['tourney_name'].str.contains("Davis Cup")].copy()


source_tournaments = (
    source_df[["Tournament"]]
    .drop_duplicates()
    .rename(columns={"Tournament": "source_tournament_name"})
)

canonical_tournaments = (
    canonical_df[["tourney_id", "tourney_name"]]
    .drop_duplicates()
    .rename(
        columns={
            "tourney_id": "canonical_tournament_id",
            "tourney_name": "canonical_tournament_name",
        }
    )
)

tournament_crosswalk = source_tournaments.merge(
    canonical_tournaments,
    left_on="source_tournament_name",
    right_on="canonical_tournament_name",
    how="left",
    validate="one_to_one",
)

tournament_crosswalk["match_method"] = "exact_name"
tournament_crosswalk["confidence"] = 1.0
tournament_crosswalk["review_flag"] = (
    tournament_crosswalk["canonical_tournament_id"].isna()
)

In [ ]:
tournament_crosswalk

In [ ]:
tournament_crosswalk["year"] = 2023
tournament_crosswalk["source"] = "tennis_data_uk"

In [ ]:
print(tournament_crosswalk)